In [3]:
import snowflake.connector
import pandas as pd

conn = snowflake.connector.connect(connection_name="criteo_svc")
s = conn.cursor().execute(
    "SELECT * FROM lift_summary ORDER BY treatment DESC"
).fetch_pandas_all()
s

,TREATMENT,N,VISITS,CONVERSIONS,VISIT_RATE,CONV_RATE
0,1,11882655,576824,36711,0.048543,0.003089
1,0,2096937,80105,4063,0.038201,0.001938


In [4]:
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

t = s[s.TREATMENT == 1].iloc[0]
c = s[s.TREATMENT == 0].iloc[0]

rows = []
for metric in ["VISITS", "CONVERSIONS"]:
    x_t, n_t = int(t[metric]), int(t["N"])
    x_c, n_c = int(c[metric]), int(c["N"])
    rate_t, rate_c = x_t / n_t, x_c / n_c

    z, p = proportions_ztest([x_t, x_c], [n_t, n_c])
    diff_lo, diff_hi = confint_proportions_2indep(x_t, n_t, x_c, n_c, compare="diff")
    ratio_lo, ratio_hi = confint_proportions_2indep(x_t, n_t, x_c, n_c, compare="ratio")

    rows.append({
        "metric": metric.lower(),
        "control_rate": rate_c,
        "treatment_rate": rate_t,
        "abs_lift": rate_t - rate_c,
        "abs_lift_ci_low": diff_lo,
        "abs_lift_ci_high": diff_hi,
        "rel_lift": rate_t / rate_c - 1,
        "rel_lift_ci_low": ratio_lo - 1,
        "rel_lift_ci_high": ratio_hi - 1,
        "z_stat": z,
        "p_value": p,
        "incremental_per_1M_users": (rate_t - rate_c) * 1_000_000,
    })

results = pd.DataFrame(rows)
results.T

,0,1
metric,visits,conversions
control_rate,0.038201,0.001938
treatment_rate,0.048543,0.003089
abs_lift,0.010342,0.001152
abs_lift_ci_low,0.010055,0.001084
abs_lift_ci_high,0.010628,0.001219
rel_lift,0.270737,0.594488
rel_lift_ci_low,0.26156,0.543532
rel_lift_ci_high,0.279967,0.646767
z_stat,65.24743,28.516529


In [5]:
results.to_csv("../data/overall_lift.csv", index=False)